In [1]:
from scipy.stats import (
    binom
)
import numpy as numpy
from seaborn import distplot
from matplotlib import pyplot
import seaborn
import random

import sys
sys.path.append('.')

In [ ]:

def inverse_plot_colorscheme():
    import cycler
    def invert(color_to_convert): 
        table = str.maketrans('0123456789abcdef', 'fedcba9876543210')
        return '#' + color_to_convert[1:].lower().translate(table).upper()
    update_dict = {}
    for key, value in pyplot.rcParams.items():
        if value == 'black':
            update_dict[key] = 'white'
        elif value == 'white':
            update_dict[key] = 'black'
    
    old_cycle = pyplot.rcParams['axes.prop_cycle']
    new_cycle = []
    for value in old_cycle:
        new_cycle.append({
            'color': invert(value['color'])
        })
    pyplot.rcParams.update(update_dict)
    pyplot.rcParams['axes.prop_cycle'] = cycler.Cycler(new_cycle)
    lec = pyplot.rcParams['legend.edgecolor']
    lec = str(1 - float(lec))
    pyplot.rcParams['legend.edgecolor'] = lec

In [3]:

inverse_plot_colorscheme()

# Часть 1. Зачем нужна прикладная статистика
## Кейс
Предположим, вы придумали идею для стартапа. Вы договорились с магазинами одежды об осуществлении доставки и наняли курьеров, которые собирают из магазинов заказы и доставляют клиентам. Предположим, **доставка** в вашем сервисе стоит **1000 рублей**, а **курьеру за работу** требуется **заплатить 500 рублей**. Казалось бы, прибыль очевидна, но внезапно вы понимаете, что **от одежды часто отказываются**, если она не подходит. Это понимают и ваши инвесторы, которых вы просите помочь с рекламой и инфраструктурой. Они **согласны помогать** вам, *если шанс отказа будет **меньше 50%***.

## Модель и наблюдения
Решая такую задачу, мы предполагаем, что есть аудитория, которая воспользуется нашим сервисом. Она называется **генеральной совокупностью**. Если мы запустим сервис на всех пользователей, то в нём будет доля успешных доставок, обозначим её $\mu$. Это некоторый параметр, который мы не знаем. Всё, что мы можем делать — это проводить эксперименты и наблюдать. Мы *не можем протестировать продукт на всех*, но можем собрать **выборку** из генеральной совокупности и пронаблюдать долю успехов. По проведённому эксперименту видим, что наблюдаемая вероятность оплаты $\frac{19}{30}=63%$.  

Но до сих пор неясно, является ли это доказательством, что истинное $\mu>50%$.

Посмотрим, почему это может не быть доказательством. Положим вероятность $\mu=0.5$ и математически смоделируем, какие могли быть статусы наших 30 заказов.

**Алгоритм моделирования**

* Так как вероятность успеха 0.5, можно подбросить монетку для моделирования того, был ли успех
* Чтобы подбросить монетку, воспользуемся функцией `random.randint`, которая **возвращает случайное целое число из диапазона**, в нашем случае от 0 до 1, где 1 означает успешную доставку
* Подкинем монетку 30 раз, чтобы получить результаты 30 доставок

In [35]:
# фиксируем последовательность рандома для воспроизводимости
random.seed(20000)

In [36]:
statuses = []
for _ in range(30):  # моделируем 30 доставок
    status = random.randint(0, 1)  # генерируем случайное число, 0 или 1
    statuses.append(status)
exp_chance = sum(statuses) / 30  # доля успехов

print('Доля успехов: {:.0%}'.format(exp_chance))

Доля успехов: 67%


Видим, что в эксперименте доля успешных заказов была даже выше 
$67$ %, но модельная вероятность была $50$ %.

Поэтому, к сожалению, **точно сказать, какой является $\mu$ в генеральной совокупности, и больше ли она 50%, нельзя**, сколько бы мы ни наблюдали за доставками. Но с помощью прикладной статистики мы введём аппарат, который позволяет принять правильное продуктовое решение, в том числе и для этой задачи.

Часть 2. Статистические гипотезы
Мы увидели, что можно получить довольно много успехов даже в случае вероятности $\mu = 0.5$. Но на самом деле для этого пришлось подбирать `seed`. Если перезапустить только ячейку с экспериментом, доля успехов получится уже меньше половины! Так может быть большая доля успехов случается очень редко, и мы готовы принять большое в качестве доказательства?

Чтобы это понять, надо ответить на вопрос:

❓ **Насколько часто может быть такое, что при $\mu=0.5$ получается большая доля успехов**?

Для этого обратимся к теории вероятностей. Успешность каждого заказа — это $\xi$ случайная величина 
 из распределения Бернулли. Параметр этого распределения, вероятность успеха, мы не знаем. $$\xi = \xi_1, \xi_2,..., \xi_{30} \sim Bern(\mu) - выборка \\ \mu - неизвестны параметр$$

## Гипотеза
При этом у нас есть продуктовая *гипотеза* о том, как устроено *распределение* в *генеральной совокупности*. Нас интересует доказательство того, что 
$\mu>0.5$. В механизме проверки гипотез в статистике рассматривают две гипотезы. `Нулевая гипотеза` говорит о том, что мы хотим *опровергнуть*. `Альтернативная гипотеза` говорит о том, что мы хотим *доказать*.
$$H_0: \mu=0.5 \\ H_1: \mu > 0.5$$


Заметим, что если в эксперименте 30 доставок, то можно смотреть **не** на **долю** успехов, **а** просто на **количество**. Переформулируем вопрос:

❓ **Насколько часто может быть такое, что при справедливости $H_0$ получается большое количество успехов**?

Количество успехов — это сумма независимых бернуллиевских величин, значит имеет биномиальное распределение. Посмотрим, как оно выглядит.


## Критерий
Только что мы придумали алгоритм, который по выборке $\xi$ либо признаёт, что мы нашли доказательство в пользу $H_1$, либо говорит, что его не нашли. Соответственно будет либо **отвергать** $H_0$, либо **не отвергать**.

Такой алгоритм называется **критерием**. Представим его как функцию $S$, которая принимает реализацию выборки и возвращает $1$, если нужно отвергнуть 
$H_0$, и $0$ иначе.

$$
    S(\xi) = 
    \begin{cases}
        1, \text{ если отвергаем } \mathsf{H}_0  \
        0, \text{ иначе}
    \end{cases}
$$,

Запишем правило, которое сформулировали выше для нашей задачи:

$$
S(\xi) = \begin{cases}
    1, \text{ если } \sum \xi_i \geqslant 20 \
    0, \text{ иначе}
\end{cases}
$$
 
Обычно сокращают запись и пишут просто правило, по которому отвергаем $H_0$

$$S = \{\sum \xi_i \geqslant 20\}$$

Обозначим $Q = \sum \xi_i$, $С = 20$,тогда критерий принимает вид
$$S = \{Q(\xi) \geqslant C\} $$


Так устроено большинство классических критериев в прикладной статистике, поэтому величинам в нём даны специальные названия. 
$Q$ называется **статистикой критерия**, $C$ — **критическим значением**.

 может быть любой функцией от выборки, которую вы считаете логичной для проверки гипотезы. В нашем случае это количество успехов, или сумма всех 
. Но вы можете выбрать и другие: максимальное значение, сумму первых 5 значений или даже просто первый элемент.